# Stream Processing on Kafka — Windowed Aggregation, Stateful Consumers

## Mental Model

This notebook demonstrates **stateful stream processing in Python** using Kafka consumers plus manual state management.

Instead of Java Kafka Streams or KSQL, we use:

- a Kafka producer to seed a telemetry alert stream
- Kafka consumers to read bounded batches
- Python dictionaries and deques as in-memory state stores
- event-time logic based on `created_at` from the alert payload

Citi narrative used throughout:

- 6,000+ API endpoints monitored for latency, throughput, and error rate
- alerts escalate through severity tiers
- endpoint storm behavior matters because bursts can cascade into wider incidents


In [ ]:
import json
import time
from collections import defaultdict, deque
from datetime import datetime, timedelta, timezone

import psycopg2
from confluent_kafka import Consumer, Producer
from confluent_kafka.admin import AdminClient, NewTopic

KAFKA_BOOTSTRAP = 'localhost:9092'
POSTGRES_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'dbname': 'de_telemetry',
    'user': 'de_admin',
    'password': 'DeAdmin2026!',
}
STREAM_TOPIC = 'citi.stream.alerts'
STREAM_EVENT_COUNT = 200

admin = AdminClient({'bootstrap.servers': KAFKA_BOOTSTRAP})

def pg_conn():
    return psycopg2.connect(**POSTGRES_CONFIG)

def ensure_topic(topic_name: str, num_partitions: int = 3, replication_factor: int = 1) -> None:
    md = admin.list_topics(timeout=10)
    if topic_name in md.topics and not md.topics[topic_name].error:
        return
    futures = admin.create_topics([
        NewTopic(topic_name, num_partitions=num_partitions, replication_factor=replication_factor)
    ])
    for _, fut in futures.items():
        try:
            fut.result()
        except Exception as exc:
            if 'already exists' not in str(exc).lower():
                raise

def parse_ts(ts_text: str) -> datetime:
    return datetime.fromisoformat(ts_text.replace('Z', '+00:00'))

def iso_utc_floor_minute(dt_obj: datetime, size_seconds: int) -> datetime:
    epoch = int(dt_obj.timestamp())
    floored = epoch - (epoch % size_seconds)
    return datetime.fromtimestamp(floored, tz=timezone.utc)

def format_table(rows, headers):
    str_rows = [[str(x) for x in row] for row in rows]
    widths = [len(str(h)) for h in headers]
    for row in str_rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(value))
    header_line = ' | '.join(str(headers[i]).ljust(widths[i]) for i in range(len(headers)))
    sep_line = '-+-'.join('-' * widths[i] for i in range(len(headers)))
    body = [' | '.join(row[i].ljust(widths[i]) for i in range(len(headers))) for row in str_rows]
    return '\n'.join([header_line, sep_line] + body)

ensure_topic(STREAM_TOPIC)
print(f'Kafka bootstrap: {KAFKA_BOOTSTRAP}')
print(f"PostgreSQL: {POSTGRES_CONFIG['host']}:{POSTGRES_CONFIG['port']}/{POSTGRES_CONFIG['dbname']}")
print(f'Topic ready: {STREAM_TOPIC}')


## Produce Telemetry Stream

We pull 200 alert records from PostgreSQL and publish them into Kafka.

Design choices:

- **topic**: `citi.stream.alerts`
- **key**: severity
- **value**: JSON payload including `created_at` so downstream logic can use event time


In [ ]:
with pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute('''
            SELECT a.alert_id,
                   a.endpoint_id,
                   a.severity,
                   a.message,
                   a.created_at,
                   e.name,
                   e.region,
                   e.status,
                   e.category
            FROM alerts a
            JOIN endpoints e
              ON e.endpoint_id = a.endpoint_id
            ORDER BY a.created_at DESC, a.alert_id DESC
            LIMIT 200
        ''')
        alert_rows = cur.fetchall()

if len(alert_rows) != STREAM_EVENT_COUNT:
    raise RuntimeError(f'Expected {STREAM_EVENT_COUNT} rows, got {len(alert_rows)}')

producer = Producer({
    'bootstrap.servers': KAFKA_BOOTSTRAP,
    'enable.idempotence': True,
    'acks': 'all',
})

produced_offsets = []

def on_delivery(err, msg):
    if err is not None:
        raise RuntimeError(str(err))
    produced_offsets.append((msg.partition(), msg.offset()))

for row in alert_rows:
    payload = {
        'alert_id': row[0],
        'endpoint_id': row[1],
        'severity': row[2],
        'message': row[3],
        'created_at': row[4].astimezone(timezone.utc).isoformat(),
        'endpoint_name': row[5],
        'region': row[6],
        'endpoint_status': row[7],
        'category': row[8],
    }
    producer.produce(
        STREAM_TOPIC,
        key=str(row[2]).encode('utf-8'),
        value=json.dumps(payload).encode('utf-8'),
        callback=on_delivery,
    )

producer.flush(15)
print('Produced 200 events to citi.stream.alerts')
print(f'Delivery reports captured: {len(produced_offsets)}')


## Tumbling Window Aggregation

This pass reads the bounded stream and groups events into **60-second tumbling windows** using the `created_at` timestamp carried in the payload.

Output shape:

- window start
- severity
- count


In [ ]:
consumer_1 = Consumer({
    'bootstrap.servers': KAFKA_BOOTSTRAP,
    'group.id': f'citi-stream-window-{int(time.time())}',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': False,
})
consumer_1.subscribe([STREAM_TOPIC])

window_counts = defaultdict(lambda: defaultdict(int))
events_pass_1 = []
target_count = STREAM_EVENT_COUNT
deadline = time.time() + 30
idle_polls = 0

while len(events_pass_1) < target_count and time.time() < deadline:
    msg = consumer_1.poll(0.5)
    if msg is None:
        idle_polls += 1
        if idle_polls >= 8 and len(events_pass_1) > 0:
            break
        continue
    if msg.error():
        raise RuntimeError(str(msg.error()))
    idle_polls = 0
    payload = json.loads(msg.value().decode('utf-8'))
    events_pass_1.append(payload)
    event_ts = parse_ts(payload['created_at']).astimezone(timezone.utc)
    window_start = iso_utc_floor_minute(event_ts, 60)
    severity = payload['severity']
    window_counts[window_start][severity] += 1

consumer_1.close()

rows = []
for window_start in sorted(window_counts.keys()):
    for severity in sorted(window_counts[window_start].keys()):
        rows.append([
            window_start.isoformat(),
            severity,
            window_counts[window_start][severity],
        ])

print(format_table(rows, ['window_start_utc', 'severity', 'count']))
print(f'Events read in pass 1: {len(events_pass_1)}')
if len(events_pass_1) == 0:
    raise RuntimeError('No events consumed in tumbling window pass')


## Stateful Count by Region

This pass joins alert events with endpoint metadata loaded from PostgreSQL, then maintains region-level counts in Python state.

This is still bounded streaming logic, but the state pattern mirrors how larger streaming engines manage keyed state.


In [ ]:
with pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT endpoint_id, region FROM endpoints')
        endpoint_region_map = {row[0]: row[1] for row in cur.fetchall()}

consumer_2 = Consumer({
    'bootstrap.servers': KAFKA_BOOTSTRAP,
    'group.id': f'citi-stream-region-{int(time.time())}',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': False,
})
consumer_2.subscribe([STREAM_TOPIC])

region_counts = defaultdict(int)
events_pass_2 = []
deadline = time.time() + 30
idle_polls = 0

while len(events_pass_2) < STREAM_EVENT_COUNT and time.time() < deadline:
    msg = consumer_2.poll(0.5)
    if msg is None:
        idle_polls += 1
        if idle_polls >= 8 and len(events_pass_2) > 0:
            break
        continue
    if msg.error():
        raise RuntimeError(str(msg.error()))
    idle_polls = 0
    payload = json.loads(msg.value().decode('utf-8'))
    events_pass_2.append(payload)
    endpoint_id = payload['endpoint_id']
    region = endpoint_region_map.get(endpoint_id, 'UNKNOWN')
    region_counts[region] += 1

consumer_2.close()

region_rows = [[region, count] for region, count in sorted(region_counts.items(), key=lambda x: (-x[1], x[0]))]
print(format_table(region_rows, ['region', 'alert_count']))
print(f'Events read in pass 2: {len(events_pass_2)}')
if len(events_pass_2) == 0:
    raise RuntimeError('No events consumed in region state pass')


## Sliding Window Alert Rate

Now we detect **hot endpoints**: any endpoint that fires more than 3 alerts within a rolling 5-minute window.

Implementation detail:

- we sort consumed events by event time
- per endpoint we maintain a deque of recent timestamps
- we evict timestamps older than 5 minutes from the current event time
- if the deque grows beyond 3, that endpoint is flagged as hot


In [ ]:
events_for_sliding = sorted(events_pass_2, key=lambda e: parse_ts(e['created_at']))
endpoint_windows = defaultdict(deque)
hot_endpoints = {}
window_size = timedelta(minutes=5)

for event in events_for_sliding:
    endpoint_id = event['endpoint_id']
    event_ts = parse_ts(event['created_at']).astimezone(timezone.utc)
    dq = endpoint_windows[endpoint_id]
    dq.append(event_ts)
    cutoff = event_ts - window_size
    while dq and dq[0] < cutoff:
        dq.popleft()
    if len(dq) > 3:
        hot_endpoints[endpoint_id] = {
            'count_in_5m': len(dq),
            'window_end_utc': event_ts.isoformat(),
        }

if hot_endpoints:
    ordered = sorted(hot_endpoints.items(), key=lambda x: (-x[1]['count_in_5m'], x[0]))
    details = [f"endpoint_id={endpoint_id}, count_in_5m={meta['count_in_5m']}, window_end_utc={meta['window_end_utc']}" for endpoint_id, meta in ordered]
    print('HOT ENDPOINTS: ' + '; '.join(details))
else:
    print('No hot endpoints detected')


## What Just Happened

You just implemented stream-processing concepts with plain Kafka consumers and Python-managed state:

- produced a bounded alert stream from PostgreSQL into Kafka
- computed **tumbling windows** by event time
- maintained **stateful counts by region**
- detected **sliding-window alert storms** per endpoint

### Micro-batch vs true streaming

This notebook behaves more like a **bounded micro-batch stream simulation**:

- we consume a fixed number of records
- we hold state in local Python memory
- there is no persistent checkpointing or distributed recovery

**True streaming engines** like Spark Structured Streaming or Flink are better for production stateful operations because they provide:

- checkpointed state
- replay and recovery guarantees
- watermarking for late data
- scalable distributed execution
- operational semantics for long-running jobs

### Watermarking concept

A **watermark** is the system's idea of how far event time has safely progressed. It helps streaming engines decide when a window can be finalized even if some late events may still arrive.

### Citi tie-in

**This pattern detects endpoint storm events before they cascade.**

That matters in a Citi-style telemetry environment because bursts of alert activity often signal a fast-moving degradation that can ripple across dependent services.
